# Model Training

## Objective

The objective of this notebook is to train and evaluate machine learning models for water potability prediction using the prepared dataset. Multiple classification algorithms will be compared, and the best-performing model will be selected for deployment in the final system.

## Theory

Machine learning classification algorithms learn patterns from historical data to predict the class of unseen samples. In this project, several supervised classification models are trained and evaluated using the prepared water potability dataset. Their performances are compared using multiple evaluation metrics before selecting the best model for deployment.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [2]:
dataset_path = "../../02_Machine Learning/Training/Potability_Model_Dataset.csv"

df = pd.read_csv(
    dataset_path,
    low_memory=False
)

print("Dataset Loaded Successfully!")

Dataset Loaded Successfully!


In [3]:
print(df.shape)

df.info()

(1023313, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1023313 entries, 0 to 1023312
Data columns (total 10 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   pH                      1023313 non-null  float64
 1   Temperature             1023313 non-null  float64
 2   Turbidity               1023313 non-null  float64
 3   Total Dissolved Solids  1023313 non-null  float64
 4   Conductivity            1023313 non-null  float64
 5   Dissolved Oxygen        85 non-null       float64
 6   Nitrate                 1023313 non-null  float64
 7   Chloride                1023313 non-null  float64
 8   Target                  1023313 non-null  int64  
 9   WQI                     0 non-null        float64
dtypes: float64(9), int64(1)
memory usage: 78.1 MB


In [4]:
df = df.drop(
    columns=[
        "Dissolved Oxygen",
        "WQI"
    ]
)

print(df.columns.tolist())

['pH', 'Temperature', 'Turbidity', 'Total Dissolved Solids', 'Conductivity', 'Nitrate', 'Chloride', 'Target']


In [5]:
df["Target"] = 1 - df["Target"].astype(int)

print(df["Target"].dtype)

int32


In [6]:
X = df.drop(columns=["Target"])

y = df["Target"]

print("Features :", X.shape)

print("Target :", y.shape)

Features : (1023313, 7)
Target : (1023313,)


In [7]:
X.head()

,pH,Temperature,Turbidity,Total Dissolved Solids,Conductivity,Nitrate,Chloride
0,6.673190,22.140587,7.947823,264.876366,395.992045,5.561756,174.687793
1,7.082763,15.583395,6.210030,264.876366,395.992045,5.561756,174.687793
2,6.123126,33.368529,9.068397,264.876366,395.992045,5.561756,174.687793
3,6.951184,22.142423,5.193508,264.876366,395.992045,5.561756,174.687793
4,8.989060,20.020028,4.350675,264.876366,395.992045,5.561756,174.687793


In [8]:
y.value_counts()

Target
0    791403
1    231910
Name: count, dtype: int64

In [9]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

print("Training Features :", X_train.shape)

print("Testing Features :", X_test.shape)

print("Training Labels :", y_train.shape)

print("Testing Labels :", y_test.shape)

Training Features : (818650, 7)
Testing Features : (204663, 7)
Training Labels : (818650,)
Testing Labels : (204663,)


In [10]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print("Feature Scaling Completed Successfully!")

Feature Scaling Completed Successfully!


In [11]:
print(X_train_scaled.shape)

print(X_test_scaled.shape)

(818650, 7)
(204663, 7)


In [12]:
from sklearn.linear_model import LogisticRegression

In [13]:
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(
    X_train_scaled,
    y_train
)

print("Logistic Regression Model Trained Successfully!")

Logistic Regression Model Trained Successfully!


In [14]:
y_pred = logistic_model.predict(
    X_test_scaled
)

In [15]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy :", round(accuracy*100,2), "%")

Accuracy : 79.04 %


In [16]:
print(

    classification_report(

        y_test,

        y_pred

    )

)

              precision    recall  f1-score   support

           0       0.80      0.97      0.88    158281
           1       0.64      0.17      0.27     46382

    accuracy                           0.79    204663
   macro avg       0.72      0.57      0.57    204663
weighted avg       0.76      0.79      0.74    204663



In [17]:
cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

[[153829   4452]
 [ 38449   7933]]


In [18]:
from sklearn.ensemble import RandomForestClassifier

In [19]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    max_depth=15,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train_scaled,
    y_train
)

print("Random Forest Model Trained Successfully!")

Random Forest Model Trained Successfully!


In [20]:
rf_pred = rf_model.predict(
    X_test_scaled
)

In [21]:
rf_accuracy = accuracy_score(
    y_test,
    rf_pred
)

print("Random Forest Accuracy :", round(rf_accuracy * 100, 2), "%")

Random Forest Accuracy : 84.89 %


In [22]:
print(classification_report(
    y_test,
    rf_pred
))

              precision    recall  f1-score   support

           0       0.92      0.88      0.90    158281
           1       0.65      0.74      0.69     46382

    accuracy                           0.85    204663
   macro avg       0.78      0.81      0.79    204663
weighted avg       0.86      0.85      0.85    204663



In [23]:
rf_cm = confusion_matrix(
    y_test,
    rf_pred
)

print(rf_cm)

[[139470  18811]
 [ 12123  34259]]


In [24]:
from sklearn.tree import DecisionTreeClassifier

In [25]:
dt_model = DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced"
)

dt_model.fit(
    X_train_scaled,
    y_train
)

print("Decision Tree Model Trained Successfully!")

Decision Tree Model Trained Successfully!


In [26]:
dt_pred = dt_model.predict(
    X_test_scaled
)

In [27]:
dt_accuracy = accuracy_score(
    y_test,
    dt_pred
)

print("Decision Tree Accuracy :", round(dt_accuracy * 100, 2), "%")

Decision Tree Accuracy : 77.22 %


In [28]:
print(classification_report(
    y_test,
    dt_pred
))

              precision    recall  f1-score   support

           0       0.86      0.85      0.85    158281
           1       0.50      0.51      0.50     46382

    accuracy                           0.77    204663
   macro avg       0.68      0.68      0.68    204663
weighted avg       0.77      0.77      0.77    204663



In [29]:
dt_cm = confusion_matrix(
    y_test,
    dt_pred
)

print(dt_cm)

[[134420  23861]
 [ 22756  23626]]


In [30]:
from sklearn.neighbors import KNeighborsClassifier

In [31]:
knn_model = KNeighborsClassifier(
    n_neighbors=5,
    n_jobs=-1
)

knn_model.fit(
    X_train_scaled,
    y_train
)

print("KNN Model Trained Successfully!")

KNN Model Trained Successfully!


In [32]:
knn_pred = knn_model.predict(
    X_test_scaled
)

In [33]:
knn_accuracy = accuracy_score(
    y_test,
    knn_pred
)

print("KNN Accuracy :", round(knn_accuracy * 100, 2), "%")

KNN Accuracy : 81.92 %


In [34]:
print(classification_report(
    y_test,
    knn_pred
))

              precision    recall  f1-score   support

           0       0.86      0.92      0.89    158281
           1       0.63      0.49      0.55     46382

    accuracy                           0.82    204663
   macro avg       0.74      0.70      0.72    204663
weighted avg       0.81      0.82      0.81    204663



In [35]:
knn_cm = confusion_matrix(
    y_test,
    knn_pred
)

print(knn_cm)

[[145015  13266]
 [ 23735  22647]]


In [36]:
from sklearn.naive_bayes import GaussianNB

In [37]:
nb_model = GaussianNB()

nb_model.fit(
    X_train_scaled,
    y_train
)

print("Gaussian Naive Bayes Model Trained Successfully!")

Gaussian Naive Bayes Model Trained Successfully!


In [38]:
nb_pred = nb_model.predict(
    X_test_scaled
)

In [39]:
nb_accuracy = accuracy_score(
    y_test,
    nb_pred
)

print("Naive Bayes Accuracy :", round(nb_accuracy * 100, 2), "%")

Naive Bayes Accuracy : 81.11 %


In [40]:
print(classification_report(
    y_test,
    nb_pred
))

              precision    recall  f1-score   support

           0       0.84      0.94      0.88    158281
           1       0.64      0.38      0.48     46382

    accuracy                           0.81    204663
   macro avg       0.74      0.66      0.68    204663
weighted avg       0.79      0.81      0.79    204663



In [41]:
nb_cm = confusion_matrix(
    y_test,
    nb_pred
)

print(nb_cm)

[[148413   9868]
 [ 28787  17595]]


In [42]:
model_results = pd.DataFrame({

    "Model":[

        "Logistic Regression",

        "Decision Tree",

        "Random Forest",

        "KNN",

        "Naive Bayes"

    ],

    "Accuracy":[

        accuracy,

        dt_accuracy,

        rf_accuracy,

        knn_accuracy,

        nb_accuracy

    ]

})

model_results["Accuracy"] = (
    model_results["Accuracy"]*100
).round(2)

model_results

,Model,Accuracy
0,Logistic Regression,79.04
1,Decision Tree,77.22
2,Random Forest,84.89
3,KNN,81.92
4,Naive Bayes,81.11


In [43]:
model_results = model_results.sort_values(

    by="Accuracy",

    ascending=False

)

model_results

,Model,Accuracy
2,Random Forest,84.89
3,KNN,81.92
4,Naive Bayes,81.11
0,Logistic Regression,79.04
1,Decision Tree,77.22


In [44]:
import os

results_folder = "../../08_Documentation/Tables"

os.makedirs(results_folder, exist_ok=True)

model_results.to_csv(

    os.path.join(

        results_folder,

        "Model_Comparison.csv"

    ),

    index=False

)

print("Model Comparison Saved Successfully!")

Model Comparison Saved Successfully!


In [45]:
import joblib

In [46]:
model_folder = "../../02_Machine Learning/Saved Models"

os.makedirs(model_folder, exist_ok=True)

joblib.dump(

    rf_model,

    os.path.join(

        model_folder,

        "Water_Potability_RF_Model.pkl"

    )

)

print("Random Forest Model Saved Successfully!")

Random Forest Model Saved Successfully!


In [47]:
joblib.dump(

    scaler,

    os.path.join(

        model_folder,

        "Feature_Scaler.pkl"

    )

)

print("Scaler Saved Successfully!")

Scaler Saved Successfully!


# Conclusion

Multiple supervised machine learning models were trained and evaluated using the prepared water potability dataset. Logistic Regression, Decision Tree, Random Forest, K-Nearest Neighbors, and Gaussian Naive Bayes were compared using accuracy, precision, recall, and F1-score. Among all evaluated models, Random Forest achieved the best overall performance with an accuracy of 84.89% and was selected as the final model for deployment in the intelligent water quality monitoring system.